# E-field distortion vs nominal PID score comparison

Separate two effects in `sel_*-updateefield` dataframe production:

1. **Recalculation bias:** CAF nominal `chi2_*` vs `chi2_*_new` (redo dE/dx + χ² with no E-field map).
2. **Pure E-field effect:** `chi2_*_new` vs `chi2_*_new_efield` (same redo, with the double-anode map).

Uses the same χ² extraction helpers as `PID.ipynb` (`avg_chi2`, `VariableConfig` binning)
and the same quality cuts as `chi2_homogeneity.ipynb` gen1 samples:
`cut_2prong_contained` → `trackscore` → `vtxdist` → `perTPC`.
1D histograms stack **trk1 + trk2** χ² values (both tracks per event).

**Expected input:** `sel_2prong-updateefield` (or `sel_mup-updateefield`) grid output with
`trk1` / `trk2` blocks containing `chi2_*`, `chi2_*_new`, and `chi2_*_new_efield`.
Set `USE_SINGLE_DF_FILE = True` to run on one merged `*.df` file instead of a directory glob.

Re-run df production after the dual-pass χ² change (`chi2_*_new` then `chi2_*_new_efield`) before comparing.


In [68]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
import glob as glob_module
import gc
import multiprocessing as mp
import pickle
from functools import partial
from os import makedirs, path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from tqdm import tqdm

import sys
sys.path.append("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")

from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.categories import PER_TPC_INCATHODE_CM
from analysis_village.numucc_1p0pi.makedf.selections import (
    cut_2prong_contained,
    cut_2prong_trackscore,
    cut_2prong_vtxdist,
    evt_has_trk1_trk2,
)
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from makedf.util import InFV, avg_chi2 as _raw_avg_chi2
from pyanalib.split_df_helpers import get_n_split

plt.style.use("presentation.mplstyle")
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

## Configuration

In [70]:
DFS_ROOT = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"

# Grid-output directory from sel_*-updateefield (edit timestamp after new submissions).
SAMPLE_DIR = path.join(DFS_ROOT, "2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar")

# Set True to read one merged *.df file instead of globbing SAMPLE_DIR.
USE_SINGLE_DF_FILE = False
SAMPLE_DF_FILE = "/exp/sbnd/data/users/munjung/cafpyana_out/sel_2prong-mc-BNB_cosmics-efieldvar_aa.df"

if USE_SINGLE_DF_FILE:
    SAMPLE_NAME = path.splitext(path.basename(SAMPLE_DF_FILE))[0]
else:
    SAMPLE_NAME = path.basename(SAMPLE_DIR)

FILENAME_STR = "sel_2prong"  # substring filter for grid *.df files (directory mode only)
N_MAX_FILES = 2000          # max number of *.df files to read (directory mode only)
N_MAX_SPLITS = None       # None = all HDF splits per file

DETECTOR = "SBND_Gen1"
TRACKSCORE_TH = 0.5
VTXDIST_TH = 1.2
PERTPC_INSET = PER_TPC_INCATHODE_CM  # 10 cm; same as chi2_homogeneity.ipynb
APPLY_2PRONG_QUAL_CUTS = True

OUT_BASE = path.join(save_fig_base_dir, "systematics-final", "Efield", "PID")
CACHE_DIR = path.join(OUT_BASE, "cache")
FIG_DIR = path.join(OUT_BASE, "plots")
CACHE_PATH = path.join(CACHE_DIR, "efield_pid_hists_v4.pkl")
makedirs(CACHE_DIR, exist_ok=True)
makedirs(FIG_DIR, exist_ok=True)

# CAF nominal, recalculated (no map), recalculated + E-field map
VERSION_COLORS = {"nominal": "C0", "recalc": "C2", "efield": "C1"}
RUN_ACCUMULATION = True
# Parallel file-level accumulation (fork Pool). Set to 1 for serial.
N_ACCUM_WORKERS = max(1, min(16, mp.cpu_count() or 4))

if USE_SINGLE_DF_FILE:
    print("Mode: single file")
    print("Sample file:", SAMPLE_DF_FILE)
else:
    print("Mode: grid directory")
    print("Sample dir:", SAMPLE_DIR)
print("Sample name:", SAMPLE_NAME)
print("Cache:", CACHE_PATH)
print("Figures:", FIG_DIR)
print("Accum workers:", N_ACCUM_WORKERS)

Mode: grid directory
Sample dir: /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar
Sample name: 2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar
Cache: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final/Efield/PID/cache/efield_pid_hists_v3.pkl
Figures: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final/Efield/PID/plots
Accum workers: 16


In [71]:
def _safe_avg_chi2(trk_df, chi2_name):
    try:
        return _raw_avg_chi2(trk_df, chi2_name).to_numpy(dtype=float)
    except Exception:
        return np.array([], dtype=float)


def _per_trk(evt_df, fn):
    """Extract one value per track from both trk1 and trk2 (concatenated).

    Matches chi2_homogeneity both-tracks 1D histograms: every event contributes
    two entries when both leading tracks are present.
    """
    out = []
    top_cols = evt_df.columns.get_level_values(0).unique()
    for trk in ("trk1", "trk2"):
        if trk not in top_cols:
            continue
        try:
            vals = np.asarray(fn(evt_df[trk]), dtype=float).ravel()
            if vals.size > 0:
                out.append(vals)
        except Exception:
            pass
    return np.concatenate(out) if out else np.array([], dtype=float)


def _per_trk_label(evt_df, trk_label, fn):
    if trk_label not in evt_df.columns.get_level_values(0):
        return np.array([], dtype=float)
    try:
        return np.asarray(fn(evt_df[trk_label]), dtype=float)
    except Exception:
        return np.array([], dtype=float)


def validate_evt_df(df, label="evt"):
    if df is None or len(df) == 0:
        raise ValueError(f"{label}: empty dataframe")
    if not evt_has_trk1_trk2(df):
        raise ValueError(
            f"{label}: missing trk1/trk2 columns. "
            "Re-run sel_*-updateefield after the updateefield wiring fix."
        )
    for score in ("chi2_muon", "chi2_muon_new", "chi2_muon_new_efield"):
        vals = _per_trk(df, lambda t, s=score: _safe_avg_chi2(t, s))
        if vals.size == 0:
            raise ValueError(
                f"{label}: could not extract {score}. "
                "Re-run sel_*-updateefield after the dual-pass χ² change."
            )
    return True


def perTPC_cut(df, incathode=PERTPC_INSET):
    """Same per-TPC cut as chi2_homogeneity.ipynb (vertex + both track ends)."""
    in_TPC1 = (
        InFV(df.slc.vertex, det="SBND_TPC1", incathode=incathode)
        & InFV(df.trk1.pfp.trk.end, det="SBND_TPC1", incathode=incathode)
        & InFV(df.trk2.pfp.trk.end, det="SBND_TPC1", incathode=incathode)
    )
    in_TPC2 = (
        InFV(df.slc.vertex, det="SBND_TPC2", incathode=incathode)
        & InFV(df.trk1.pfp.trk.end, det="SBND_TPC2", incathode=incathode)
        & InFV(df.trk2.pfp.trk.end, det="SBND_TPC2", incathode=incathode)
    )
    return in_TPC1 | in_TPC2


def apply_optional_qual_cuts(df):
    """Apply the same quality cuts as chi2_homogeneity gen1 samples.

    Order: contained → trackscore → vtxdist → perTPC.
    """
    if not APPLY_2PRONG_QUAL_CUTS:
        return df
    if not evt_has_trk1_trk2(df):
        return df.iloc[0:0]
    df = cut_2prong_contained(df, det=DETECTOR)
    df = cut_2prong_trackscore(df, TRACKSCORE_TH)
    df = cut_2prong_vtxdist(df, VTXDIST_TH)
    if len(df) == 0:
        return df
    return df.loc[perTPC_cut(df)]


VAR_DEFS = {
    # extract() stacks trk1 + trk2 χ² values into one 1D histogram.
    "chi2_mu_nom": {
        "label": VariableConfig.chi2_mu().var_labels[0] + " (CAF nominal)",
        "bins": np.asarray(VariableConfig.chi2_mu().bins),
        "extract": lambda df: _per_trk(df, lambda t: _safe_avg_chi2(t, "chi2_muon")),
    },
    "chi2_mu_new": {
        "label": VariableConfig.chi2_mu().var_labels[0] + " (recalc.)",
        "bins": np.asarray(VariableConfig.chi2_mu().bins),
        "extract": lambda df: _per_trk(df, lambda t: _safe_avg_chi2(t, "chi2_muon_new")),
    },
    "chi2_mu_efield": {
        "label": VariableConfig.chi2_mu().var_labels[0] + " (recalc. + E-field)",
        "bins": np.asarray(VariableConfig.chi2_mu().bins),
        "extract": lambda df: _per_trk(df, lambda t: _safe_avg_chi2(t, "chi2_muon_new_efield")),
    },
    "chi2_p_nom": {
        "label": VariableConfig.chi2_proton().var_labels[0] + " (CAF nominal)",
        "bins": np.asarray(VariableConfig.chi2_proton().bins),
        "extract": lambda df: _per_trk(df, lambda t: _safe_avg_chi2(t, "chi2_proton")),
    },
    "chi2_p_new": {
        "label": VariableConfig.chi2_proton().var_labels[0] + " (recalc.)",
        "bins": np.asarray(VariableConfig.chi2_proton().bins),
        "extract": lambda df: _per_trk(df, lambda t: _safe_avg_chi2(t, "chi2_proton_new")),
    },
    "chi2_p_efield": {
        "label": VariableConfig.chi2_proton().var_labels[0] + " (recalc. + E-field)",
        "bins": np.asarray(VariableConfig.chi2_proton().bins),
        "extract": lambda df: _per_trk(df, lambda t: _safe_avg_chi2(t, "chi2_proton_new_efield")),
    },
}

# (ref_key, var_key, tag, ref_legend, var_legend, ratio_ylabel, ref_color, var_color)
PAIR_DEFS = [
    # Pure E-field effect (same recalculation path, with vs without map).
    (
        "chi2_mu_new", "chi2_mu_efield", "chi2_mu_efield_vs_recalc",
        "Recalc. χ²", "Recalc. χ² + E-field", "E-field / Recalc.", "recalc", "efield",
    ),
    (
        "chi2_p_new", "chi2_p_efield", "chi2_p_efield_vs_recalc",
        "Recalc. χ²", "Recalc. χ² + E-field", "E-field / Recalc.", "recalc", "efield",
    ),
    # Recalculation bias (CAF stored χ² vs redo with no E-field map).
    (
        "chi2_mu_nom", "chi2_mu_new", "chi2_mu_recalc_vs_caf",
        "CAF nominal χ²", "Recalc. χ²", "Recalc. / CAF", "nominal", "recalc",
    ),
    (
        "chi2_p_nom", "chi2_p_new", "chi2_p_recalc_vs_caf",
        "CAF nominal χ²", "Recalc. χ²", "Recalc. / CAF", "nominal", "recalc",
    ),
]

# Detector octants from slc.vertex (same dividers as chi2_homogeneity.ipynb).
OCTANT_TITLES = [
    "x<0, y<0, z<250",
    "x<0, y<0, z>=250",
    "x<0, y>=0, z<250",
    "x<0, y>=0, z>=250",
    "x>=0, y<0, z<250",
    "x>=0, y<0, z>=250",
    "x>=0, y>=0, z<250",
    "x>=0, y>=0, z>=250",
]


def get_octant_dfs(df):
    """Split df into 8 octants on slc.vertex with dividers x=0, y=0, z=250."""
    octants = []
    for i in range(8):
        mask_x = (df.slc.vertex.x >= 0) if ((i >> 2) & 1) else (df.slc.vertex.x < 0)
        mask_y = (df.slc.vertex.y >= 0) if ((i >> 1) & 1) else (df.slc.vertex.y < 0)
        mask_z = (df.slc.vertex.z >= 250) if ((i >> 0) & 1) else (df.slc.vertex.z < 250)
        octants.append(df[mask_x & mask_y & mask_z])
    return octants


print("Variables:", list(VAR_DEFS.keys()), "(1D fills use trk1 + trk2)")
print(
    "Cuts: contained(%s) → trackscore>%.2f → vtxdist<%.2f → perTPC(inset=%s cm)"
    % (DETECTOR, TRACKSCORE_TH, VTXDIST_TH, PERTPC_INSET)
)
print("Octants:", OCTANT_TITLES)

Variables: ['chi2_mu_nom', 'chi2_mu_efield', 'chi2_p_nom', 'chi2_p_efield'] (1D fills use trk1 + trk2)
Cuts: contained(SBND_Gen1) → trackscore>0.50 → vtxdist<1.20 → perTPC(inset=10 cm)
Octants: ['x<0, y<0, z<250', 'x<0, y<0, z>=250', 'x<0, y>=0, z<250', 'x<0, y>=0, z>=250', 'x>=0, y<0, z<250', 'x>=0, y<0, z>=250', 'x>=0, y>=0, z<250', 'x>=0, y>=0, z>=250']


In [72]:
def get_sample_df_files():
    """Return *.df paths for the configured sample (directory glob or single file)."""
    if USE_SINGLE_DF_FILE:
        if not SAMPLE_DF_FILE:
            raise ValueError("USE_SINGLE_DF_FILE=True but SAMPLE_DF_FILE is not set")
        if not path.isfile(SAMPLE_DF_FILE):
            raise FileNotFoundError(f"SAMPLE_DF_FILE not found: {SAMPLE_DF_FILE}")
        return [SAMPLE_DF_FILE]

    pattern = path.join(SAMPLE_DIR, f"*{FILENAME_STR}*.df")
    files = sorted(glob_module.glob(pattern))[: int(N_MAX_FILES)]
    if not files:
        raise FileNotFoundError(f"No files for pattern {pattern}")
    return files


def _fill_hists_from_split(df, var_defs, hists):
    if df is None or len(df) == 0:
        return 0
    n = len(df)
    for var_name, cfg in var_defs.items():
        try:
            vals = cfg["extract"](df)
        except Exception:
            continue
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue
        eps = (cfg["bins"][-1] - cfg["bins"][0]) * 1e-9
        vals = np.clip(vals, cfg["bins"][0], cfg["bins"][-1] - eps)
        counts, _ = np.histogram(vals, bins=cfg["bins"])
        hists[var_name] += counts
    return n


def _read_evt_split(file_path, split_idx):
    return pd.read_hdf(file_path, key=f"evt_{split_idx}")


def _empty_hists(var_defs):
    return {v: np.zeros(len(cfg["bins"]) - 1, dtype=float) for v, cfg in var_defs.items()}


def _accumulate_one_file(fpath, n_max_splits=None):
    """Process one *.df file → per-file hist payloads (Pool worker; uses fork globals)."""
    var_defs = VAR_DEFS
    if n_max_splits is None:
        n_max_splits = N_MAX_SPLITS

    hists = _empty_hists(var_defs)
    octant_hists = [_empty_hists(var_defs) for _ in range(len(OCTANT_TITLES))]
    n_evts = 0
    n_evts_oct = np.zeros(len(OCTANT_TITLES), dtype=int)

    try:
        n_split = get_n_split(fpath)
    except Exception as exc:
        return {"ok": False, "file": fpath, "error": str(exc)}

    n_use = n_split if n_max_splits is None else min(n_split, int(n_max_splits))
    for i in range(n_use):
        try:
            df = _read_evt_split(fpath, i)
        except Exception:
            continue
        df = apply_optional_qual_cuts(df)
        n_evts += _fill_hists_from_split(df, var_defs, hists)
        if len(df) and "slc" in df.columns.get_level_values(0):
            try:
                for oi, odf in enumerate(get_octant_dfs(df)):
                    n_evts_oct[oi] += _fill_hists_from_split(odf, var_defs, octant_hists[oi])
            except Exception:
                pass
        del df
    gc.collect()
    return {
        "ok": True,
        "file": fpath,
        "hists": hists,
        "n_evts": n_evts,
        "octant_hists": octant_hists,
        "n_evts_oct": n_evts_oct,
    }


def _reduce_accum_results(results, var_defs):
    hists = _empty_hists(var_defs)
    octant_hists = [_empty_hists(var_defs) for _ in range(len(OCTANT_TITLES))]
    n_evts = 0
    n_evts_oct = np.zeros(len(OCTANT_TITLES), dtype=int)
    n_fail = 0
    for res in results:
        if not res.get("ok", False):
            n_fail += 1
            print(f"skip {path.basename(res.get('file', '?'))}: {res.get('error')}")
            continue
        n_evts += int(res["n_evts"])
        n_evts_oct += np.asarray(res["n_evts_oct"], dtype=int)
        for k in hists:
            hists[k] += res["hists"][k]
        for oi in range(len(OCTANT_TITLES)):
            for k in octant_hists[oi]:
                octant_hists[oi][k] += res["octant_hists"][oi][k]
    if n_fail:
        print(f"Failed files: {n_fail}")
    return hists, n_evts, octant_hists, n_evts_oct


def accumulate_from_sample(var_defs, n_max_splits, n_workers=None):
    files = get_sample_df_files()
    desc = path.basename(files[0]) if USE_SINGLE_DF_FILE else path.basename(SAMPLE_DIR)
    if n_workers is None:
        n_workers = N_ACCUM_WORKERS
    n_workers = max(1, min(int(n_workers), len(files)))

    worker = partial(_accumulate_one_file, n_max_splits=n_max_splits)

    if n_workers == 1:
        results = [worker(fpath) for fpath in tqdm(files, desc=desc)]
        return _reduce_accum_results(results, var_defs)

    # fork: workers inherit notebook globals (VAR_DEFS extractors, cut helpers, …)
    ctx = mp.get_context("fork")
    with ctx.Pool(processes=n_workers) as pool:
        results = list(
            tqdm(
                pool.imap_unordered(worker, files),
                total=len(files),
                desc=f"{desc} [{n_workers} workers]",
            )
        )
    return _reduce_accum_results(results, var_defs)


def save_hists(cache_path, hists, n_evts, var_defs, sample_name, octant_hists=None, n_evts_oct=None):
    payload = {
        "version": 3,
        "sample": sample_name,
        "var_defs": {
            k: {"label": v["label"], "bins": np.asarray(v["bins"])}
            for k, v in var_defs.items()
        },
        "hists": hists,
        "n_evts": n_evts,
        "octant_hists": octant_hists,
        "n_evts_oct": None if n_evts_oct is None else np.asarray(n_evts_oct),
        "octant_titles": list(OCTANT_TITLES),
    }
    with open(cache_path, "wb") as fh:
        pickle.dump(payload, fh, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved cache → {cache_path}")
    return payload


def load_hists(cache_path):
    with open(cache_path, "rb") as fh:
        payload = pickle.load(fh)
    n_oct = len(payload.get("octant_hists") or [])
    print(
        f"Loaded cache: sample={payload['sample']!r}, "
        f"{len(payload['var_defs'])} variables, {payload['n_evts']:,} events"
        + (f", {n_oct} octants" if n_oct else "")
    )
    return payload

## Inspect loaded dataframe

Load the first `evt_0` split from the grid output and print column layout before histogram accumulation. Useful to confirm `trk1`/`trk2` and `chi2_*` / `chi2_*_new` are present (does not apply quality cuts or raise on missing tracks).

In [73]:
def print_evt_df_summary(df, label="evt"):
    top_cols = df.columns.get_level_values(0).unique()
    print(f"{label}: {len(df):,} rows, index levels={df.index.names}")
    print(f"  top-level columns ({len(top_cols)}): {list(top_cols)}")
    for trk in ("trk1", "trk2", "slc", "mc"):
        present = trk in top_cols
        print(f"  {trk}: {'present' if present else 'MISSING'}")
        if present and trk.startswith("trk"):
            sub = df[trk].columns
            chi2_cols = [c for c in sub if "chi2" in str(c).lower()]
            suffix = "..." if len(chi2_cols) > 8 else ""
            print(f"    {len(sub)} sub-columns; chi2-related: {chi2_cols[:8]}{suffix}")
    print(f"  evt_has_trk1_trk2: {evt_has_trk1_trk2(df)}")


try:
    _inspect_files = get_sample_df_files()
except FileNotFoundError as exc:
    print(exc)
    loaded_evt_df = None
else:
    _inspect_path = _inspect_files[0]
    loaded_evt_df = _read_evt_split(_inspect_path, 0)
    print(f"File: {_inspect_path}")
    print("HDF key: evt_0\n")
    print_evt_df_summary(loaded_evt_df, label=SAMPLE_NAME)
    print("\nFirst rows:")
    loaded_evt_df.head()

File: /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar/sel_2prong-mc-BNB_cosmics-efieldvar_0.df
HDF key: evt_0

2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar: 741 rows, index levels=['__ntuple', 'entry', 'rec.slc..index']
  top-level columns (9): ['slc', 'n_trks', 'n_good_trks', 'nocut_trk1', 'trk1', 'nocut_trk2', 'trk2', 'rec.mc.nu..index', 'mc']
  trk1: present
    91 sub-columns; chi2-related: [('pfp', 'trk', 'chi2pid', 'I0', 'pid_ndof', ''), ('pfp', 'trk', 'chi2pid', 'I0', 'chi2_muon', ''), ('pfp', 'trk', 'chi2pid', 'I0', 'chi2_proton', ''), ('pfp', 'trk', 'chi2pid', 'I0', 'pida', ''), ('pfp', 'trk', 'chi2pid', 'I1', 'pid_ndof', ''), ('pfp', 'trk', 'chi2pid', 'I1', 'chi2_muon', ''), ('pfp', 'trk', 'chi2pid', 'I1', 'chi2_proton', ''), ('pfp', 'trk', 'chi2pid', 'I1', 'pida', '')]...
  trk2: present
    91 sub-columns; chi2-related: [('pfp', 'trk', 'chi2pid', 'I0', 'pid_ndof', ''), ('pfp', 'trk', 'chi2pid', 'I0', 'chi2_m

In [74]:
loaded_evt_df

slc                          \
                              is_clear_cosmic      vertex               
                                                        x           y   
                                                                        
                                                                        
                                                                        
                                                                        
__ntuple entry rec.slc..index                                           
6        8     3                            0  151.695496   -9.043008   
         9     0                            0  101.540382  156.087021   
         12    0                            0 -182.793060   90.577148   
               1                            0 -172.807755 -136.376114   
         19    0                            0 -159.218887 -182.726624   
..                                        ...         ...         ...   
186      7     0                            0   93.125572  116.194572   
         10    1                            0  -23.897139  -93.257904   
         11    0                            0  108.545967  -61.172714   
187      8     0                            0  -32.939220   68.782150   
         9     0                            0  -75.077927  -28.160860   

                                                                           \
                                          self    tmatch                    
                                        z            eff       pur    idx   
                                                                            
                                                                            
                                                                            
                                                                            
__ntuple entry rec.slc..index                                               
6        8     3               352.324249  110       NaN       NaN -999.0   
         9     0                61.865826   56  0.901585  0.915081    0.0   
         12    0               103.870705   40  0.627491  0.957628    0.0   
               1               443.868439   41  0.334707  0.909580    NaN   
         19    0               373.481384   55  0.974825  0.977184    0.0   
..                                    ...  ...       ...       ...    ...   
186      7     0               145.011154   60  0.964809  0.954518    0.0   
         10    1               295.757538   41  0.862298  0.842497    0.0   
         11    0               160.442673   17  0.954137  0.855689    0.0   
187      8     0               203.414215   72  0.929289  0.891866    0.0   
         9     0               186.396500   74  0.863545  0.958431    0.0   

                                                      ...        mc            \
                              producer          nuid  ...         p             
                                       crlongtrkdiry  ...       dir             
                                                      ...         x         y   
                                                      ...                       
                                                      ...                       
                                                      ...                       
__ntuple entry rec.slc..index                         ...                       
6        8     3                     0     -0.263111  ...       NaN       NaN   
         9     0                     0     -0.118469  ... -0.876162 -0.243430   
         12    0                     0     -0.427406  ...  0.235809  0.821572   
               1                     0     -0.603083  ...       NaN       NaN   
         19    0                     0     -0.087542  ...  0.865333 -0.480119   
..                                 ...           ...  ...       ...       ...   
186      7     0                     0     -0.890541  ..

In [75]:
if RUN_ACCUMULATION:
    hists, n_evts, octant_hists, n_evts_oct = accumulate_from_sample(VAR_DEFS, N_MAX_SPLITS)
    print(f"events after cuts: {n_evts:,}")
    for title, n in zip(OCTANT_TITLES, n_evts_oct):
        print(f"  {title}: {n:,}")
    save_hists(
        CACHE_PATH, hists, n_evts, VAR_DEFS, SAMPLE_NAME,
        octant_hists=octant_hists, n_evts_oct=n_evts_oct,
    )
else:
    print("Skipping accumulation; will load cache in next cell.")

2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar [16 workers]: 100%|██████████| 1956/1956 [16:04<00:00,  2.03it/s]  

skip sel_2prong-mc-BNB_cosmics-efieldvar_1061.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1085.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1156.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1162.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1181.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1203.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1267.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1304.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1319.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1337.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_135.df: 'No object named split in the file'
skip sel_2prong-mc-BNB_cosmics-efieldvar_1371.df: 'No o

In [76]:
if not RUN_ACCUMULATION:
    _payload = load_hists(CACHE_PATH)
    hists = _payload["hists"]
    n_evts = _payload["n_evts"]
    octant_hists = _payload.get("octant_hists")
    n_evts_oct = _payload.get("n_evts_oct")
    PLOT_VAR_DEFS = _payload["var_defs"]
else:
    PLOT_VAR_DEFS = {
        k: {"label": v["label"], "bins": np.asarray(v["bins"])}
        for k, v in VAR_DEFS.items()
    }

print(f"{SAMPLE_NAME}: {n_evts:,} events")
if octant_hists is not None and n_evts_oct is not None:
    print(f"octant events: {dict(zip(OCTANT_TITLES, map(int, n_evts_oct)))}")
else:
    print("No octant histograms in cache; re-run accumulation to enable octant plots.")

2026_07_10_073251__sel_2prong-mc-BNB_cosmics-efieldvar: 187,006 events
octant events: {'x<0, y<0, z<250': 36152, 'x<0, y<0, z>=250': 20234, 'x<0, y>=0, z<250': 30004, 'x<0, y>=0, z>=250': 9296, 'x>=0, y<0, z<250': 34201, 'x>=0, y<0, z>=250': 19149, 'x>=0, y>=0, z<250': 29089, 'x>=0, y>=0, z>=250': 8881}


In [ ]:
def save_fig(fig, name, fig_dir=FIG_DIR, dpi=150):
    makedirs(fig_dir, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(path.join(fig_dir, f"{name}.{ext}"), bbox_inches="tight", dpi=dpi)
    print(f"  saved {name}.pdf / .png")


def _axis_label(var_defs, nom_key):
    """X-axis label without parenthetical suffixes like '(CAF nominal)'."""
    return var_defs[nom_key]["label"].split("(")[0].strip()


def _draw_pair_1d(
    ax_top,
    ax_bot,
    ref_key,
    var_key,
    var_defs,
    hists,
    ref_label="Reference χ²",
    var_label="Varied χ²",
    ratio_ylabel="Var. / Ref.",
    ref_color="nominal",
    var_color="efield",
    normalize=False,
    title=None,
    xlabel=True,
    ylabel=True,
    legend=True,
    legend_fontsize=12,
    ax_ylim_ratio=(0.0, 2.0),
):
    bins = np.asarray(var_defs[ref_key]["bins"])
    ref = np.asarray(hists[ref_key], dtype=float)
    var = np.asarray(hists[var_key], dtype=float)
    if normalize:
        ref_tot, var_tot = ref.sum(), var.sum()
        if ref_tot > 0:
            ref = ref / ref_tot
        if var_tot > 0:
            var = var / var_tot

    c_ref = VERSION_COLORS[ref_color]
    c_var = VERSION_COLORS[var_color]
    ax_top.step(
        bins, np.append(ref, ref[-1]), where="post",
        label=ref_label, color=c_ref, linewidth=1.5,
    )
    ax_top.step(
        bins, np.append(var, var[-1]), where="post",
        label=var_label, color=c_var, linewidth=1.5,
    )
    ax_top.set_xlim(bins[0], bins[-1])
    if ylabel:
        ax_top.set_ylabel("Tracks")
    if legend:
        ax_top.legend(frameon=True, fontsize=legend_fontsize)
    if title:
        ax_top.set_title(title, fontsize=10)

    if ax_bot is not None:
        with np.errstate(divide="ignore", invalid="ignore"):
            rat = np.where(ref > 0, var / ref, np.nan)
        ax_bot.axhline(1.0, color="k", linewidth=0.8, linestyle="--")
        ax_bot.step(bins, np.append(rat, rat[-1]), where="post", color=c_var)
        ax_bot.set_xlim(bins[0], bins[-1])
        if ylabel:
            ax_bot.set_ylabel(ratio_ylabel)
        if xlabel:
            ax_bot.set_xlabel(_axis_label(var_defs, ref_key))
        ax_bot.set_ylim(*ax_ylim_ratio)
    elif xlabel:
        ax_top.set_xlabel(_axis_label(var_defs, ref_key))


def plot_pair_1d(
    sample_name,
    ref_key,
    var_key,
    var_defs,
    hists,
    ref_label="Reference χ²",
    var_label="Varied χ²",
    ratio_ylabel="Var. / Ref.",
    ref_color="nominal",
    var_color="efield",
    ratio=True,
    normalize=False,
):
    if ratio:
        fig, (ax_top, ax_bot) = plt.subplots(
            2, 1, figsize=(7, 6), sharex=True, gridspec_kw={"height_ratios": [4, 1]}
        )
        fig.subplots_adjust(hspace=0.08)
    else:
        fig, ax_top = plt.subplots(figsize=(7, 4.5))
        ax_bot = None

    _draw_pair_1d(
        ax_top, ax_bot, ref_key, var_key, var_defs, hists,
        ref_label=ref_label, var_label=var_label, ratio_ylabel=ratio_ylabel,
        ref_color=ref_color, var_color=var_color,
        normalize=normalize, title=None, legend_fontsize=12,
    )
    fig.tight_layout()
    return fig


def plot_pair_1d_octants(
    sample_name,
    ref_key,
    var_key,
    var_defs,
    octant_hists,
    ref_label="Reference χ²",
    var_label="Varied χ²",
    ratio_ylabel="Var. / Ref.",
    ref_color="nominal",
    var_color="efield",
    n_evts_oct=None,
    ratio=True,
    normalize=False,
    ax_ylim_ratio=(0.0, 2.0),
):
    """2x4 grid of paired χ² comparisons, one panel per detector octant.

    Octants follow chi2_homogeneity.ipynb: dividers at x=0, y=0, z=250 on slc.vertex.
    """
    if octant_hists is None or len(octant_hists) != len(OCTANT_TITLES):
        raise ValueError("octant_hists must contain 8 histograms (one per octant)")

    n_oct = len(OCTANT_TITLES)
    n_cols = 4
    n_row_panels = 2
    n_gs_rows = (2 * n_row_panels) if ratio else n_row_panels
    height_ratios = ([3, 1] * n_row_panels) if ratio else [1] * n_row_panels

    fig, axes = plt.subplots(
        n_gs_rows,
        n_cols,
        figsize=(5.0 * n_cols, 4.5 * n_row_panels),
        sharex=True,
        gridspec_kw={"height_ratios": height_ratios, "hspace": 0.35, "wspace": 0.30},
    )
    axes = np.atleast_2d(axes)

    for i in range(n_oct):
        row_p, col = divmod(i, n_cols)
        if ratio:
            ax_top = axes[2 * row_p, col]
            ax_bot = axes[2 * row_p + 1, col]
        else:
            ax_top = axes[row_p, col]
            ax_bot = None

        _draw_pair_1d(
            ax_top, ax_bot, ref_key, var_key, var_defs, octant_hists[i],
            ref_label=ref_label, var_label=var_label, ratio_ylabel=ratio_ylabel,
            ref_color=ref_color, var_color=var_color,
            normalize=normalize,
            title=OCTANT_TITLES[i],
            xlabel=(row_p == n_row_panels - 1),
            ylabel=(col == 0),
            legend=(i == 0),
            legend_fontsize=11,
            ax_ylim_ratio=ax_ylim_ratio,
        )
        ax_top.tick_params(axis="both", labelsize=8)
        if ax_bot is not None:
            ax_bot.tick_params(axis="both", labelsize=8)

    fig.tight_layout()
    return fig


for ref_key, var_key, tag, ref_lab, var_lab, ratio_ylab, ref_c, var_c in PAIR_DEFS:
    fig = plot_pair_1d(
        SAMPLE_NAME, ref_key, var_key, PLOT_VAR_DEFS, hists,
        ref_label=ref_lab, var_label=var_lab, ratio_ylabel=ratio_ylab,
        ref_color=ref_c, var_color=var_c, ratio=True,
    )
    save_fig(fig, f"pid_1d_{tag}")
    plt.show()
    plt.close(fig)

if octant_hists is not None:
    for ref_key, var_key, tag, ref_lab, var_lab, ratio_ylab, ref_c, var_c in PAIR_DEFS:
        fig = plot_pair_1d_octants(
            SAMPLE_NAME, ref_key, var_key, PLOT_VAR_DEFS, octant_hists,
            ref_label=ref_lab, var_label=var_lab, ratio_ylabel=ratio_ylab,
            ref_color=ref_c, var_color=var_c,
            n_evts_oct=n_evts_oct, ratio=True,
        )
        save_fig(fig, f"pid_1d_{tag}_octants")
        plt.show()
        plt.close(fig)
else:
    print("Skipping octant plots (no octant_hists).")


## 2D PID planes (μ vs p χ²)

CAF nominal, recalculated (no map), and recalculated + E-field (`PID.ipynb` style).


In [ ]:
def load_evt_subset(n_files=5, n_splits=1):
    """Load evt splits from the configured sample for in-memory plots."""
    files = get_sample_df_files()[:n_files]
    frames = []
    for fpath in files:
        n_split = min(get_n_split(fpath), n_splits)
        for i in range(n_split):
            frames.append(_read_evt_split(fpath, i))
    if not frames:
        raise FileNotFoundError(f"No evt splits found for configured sample ({SAMPLE_NAME})")
    df = pd.concat(frames, axis=0, sort=False)
    validate_evt_df(df, label=SAMPLE_NAME)
    return apply_optional_qual_cuts(df)


def plot_pid_plane_2d(evt_df, trk_label, chi2_mu_tag, chi2_p_tag, title_prefix=""):
    mu = _per_trk_label(evt_df, trk_label, lambda t: _safe_avg_chi2(t, chi2_mu_tag))
    pp = _per_trk_label(evt_df, trk_label, lambda t: _safe_avg_chi2(t, chi2_p_tag))
    mask = np.isfinite(mu) & np.isfinite(pp)
    mu, pp = mu[mask], pp[mask]

    cfg_mu = VariableConfig.chi2_mu()
    cfg_p = VariableConfig.chi2_proton()

    fig, ax = plt.subplots(figsize=(6, 5))
    h = ax.hist2d(
        mu, pp,
        bins=[cfg_mu.bins, cfg_p.bins],
        cmap="viridis",
        norm=LogNorm(),
    )
    fig.colorbar(h[3], ax=ax, label="entries")
    ax.set_xlabel(cfg_mu.var_labels[0])
    ax.set_ylabel(cfg_p.var_labels[0])
    ax.set_title(f"{title_prefix}{trk_label}: {chi2_mu_tag} vs {chi2_p_tag}")
    fig.tight_layout()
    return fig


evt_df = load_evt_subset(n_files=5, n_splits=1)
print(f"loaded {len(evt_df):,} events from {SAMPLE_NAME}")

for trk in ("trk1", "trk2"):
    for chi2_mu_tag, chi2_p_tag, tag in (
        ("chi2_muon", "chi2_proton", "caf"),
        ("chi2_muon_new", "chi2_proton_new", "recalc"),
        ("chi2_muon_new_efield", "chi2_proton_new_efield", "efield"),
    ):
        fig = plot_pid_plane_2d(
            evt_df, trk, chi2_mu_tag, chi2_p_tag,
            title_prefix=f"{SAMPLE_NAME} / ",
        )
        save_fig(fig, f"pid_2d_{trk}_{tag}")
        plt.show()
        plt.close(fig)


## Per-track Δχ²

Two comparisons, each stacking **trk1 + trk2**:

1. E-field − recalc: `chi2_*_new_efield − chi2_*_new` (pure E-field shift)
2. Recalc − CAF: `chi2_*_new − chi2_*` (recalculation bias)


In [ ]:
def paired_delta_chi2(evt_df, trk_label, a_name, b_name):
    """Return b − a for one track block."""
    a = _per_trk_label(evt_df, trk_label, lambda t, n=a_name: _safe_avg_chi2(t, n))
    b = _per_trk_label(evt_df, trk_label, lambda t, n=b_name: _safe_avg_chi2(t, n))
    mask = np.isfinite(a) & np.isfinite(b)
    return b[mask] - a[mask]


def paired_delta_chi2_both_trks(evt_df, a_name, b_name):
    """Δχ² for trk1 and trk2 concatenated (one entry per track)."""
    parts = [paired_delta_chi2(evt_df, trk, a_name, b_name) for trk in ("trk1", "trk2")]
    parts = [p for p in parts if p.size]
    return np.concatenate(parts) if parts else np.array([], dtype=float)


evt_df = load_evt_subset(n_files=8, n_splits=1)

DELTA_DEFS = [
    # (par, tag, a_col, b_col, xlabel, color_key)
    (
        "muon", "chi2_mu_efield_minus_recalc",
        "chi2_muon_new", "chi2_muon_new_efield",
        r"$\chi^2_{\mu\mathrm{,\,efield}} - \chi^2_{\mu\mathrm{,\,recalc}}$",
        "efield",
    ),
    (
        "proton", "chi2_p_efield_minus_recalc",
        "chi2_proton_new", "chi2_proton_new_efield",
        r"$\chi^2_{p\mathrm{,\,efield}} - \chi^2_{p\mathrm{,\,recalc}}$",
        "efield",
    ),
    (
        "muon", "chi2_mu_recalc_minus_caf",
        "chi2_muon", "chi2_muon_new",
        r"$\chi^2_{\mu\mathrm{,\,recalc}} - \chi^2_{\mu\mathrm{,\,CAF}}$",
        "recalc",
    ),
    (
        "proton", "chi2_p_recalc_minus_caf",
        "chi2_proton", "chi2_proton_new",
        r"$\chi^2_{p\mathrm{,\,recalc}} - \chi^2_{p\mathrm{,\,CAF}}$",
        "recalc",
    ),
]

for _par, tag, a_name, b_name, xlabel, color_key in DELTA_DEFS:
    delta = paired_delta_chi2_both_trks(evt_df, a_name, b_name)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    if delta.size:
        ax.hist(delta, bins=600, histtype="step", color=VERSION_COLORS[color_key], linewidth=1.5)
    ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Tracks")
    ax.set_xlim(-10, 100)
    fig.tight_layout()
    save_fig(fig, f"pid_delta_{tag}")
    plt.show()
    plt.close(fig)
